In [15]:
!pip install transformers datasets sentencepiece
!pip install accelerate

In [16]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('api_huggingface'))

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch, json

device = 0 if torch.cuda.is_available() else -1
print("using:", "GPU" if device == 0 else "CPU")

model_name = "sarvamai/sarvam-2b-v0.5"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16 if device==0 else torch.float32).to("cuda" if device==0 else "cpu")

generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)

✅ Using: GPU


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [18]:
prompts = [
    "एक परामर्श संवाद लिखिए जिसमें क्लाइंट परीक्षा की चिंता के बारे में बात करता है और काउंसलर सहानुभूति दिखाता है।",
    "एक बातचीत लिखिए जिसमें क्लाइंट अपने परिवार से झगड़ों को लेकर दुखी है और काउंसलर समझदारी और समर्थन देता है।",
    "क्लाइंट अकेलापन महसूस कर रहा है, और काउंसलर उससे भावनात्मक जुड़ाव दिखाता है।",
    "क्लाइंट काम के तनाव से परेशान है, और काउंसलर सकारात्मक प्रतिक्रिया देता है।",
    "क्लाइंट मानसिक थकान की शिकायत करता है, और काउंसलर सहानुभूतिपूर्ण शब्दों से मदद करता है।"
]

In [19]:
dialogues = []
for i, p in enumerate(prompts):
    outs = generator(p, max_length=180, temperature=0.8, top_p=0.9, num_return_sequences=4, do_sample=True)
    for j, o in enumerate(outs):
        dialogues.append({"id": f"hindi_{i}_{j}", "language": "Hindi", "dialogue": o["generated_text"].strip()})

with open("/content/drive/MyDrive/hygieia/data/hindi_sarvam.json", "w", encoding="utf-8") as f:
    json.dump(dialogues, f, ensure_ascii=False, indent=2)

print("✅ Saved", len(dialogues), "Hindi dialogues")

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


generated 50 dialogues
